# Data Loading

The `PupilMeasurement` object can be created by loading data from a file, internal data can be used for testing, or created from an array-like.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import piprkit

## Loading built-in data

Piprkit provides a pre-simulated dataset as well as some data measured in the field.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 2))

pm_red = piprkit.load_simulated_pupil(blue_or_red='red')
pm_red.plot(ax=ax, color='red', label='Red')

pm_blue = piprkit.load_simulated_pupil(blue_or_red='blue')
pm_blue.plot(ax=ax, color='blue', label='Blue')

pm_blue.plot_light_stimulus(ax=ax, color='yellow', alpha=0.3)

ax.legend()
plt.show()

In [ ]:
ps = piprkit.load_real_series()

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
ps.plot(ax=ax, color='darkblue', label='Real')
ps.plot_light_stimulus(ax=ax, color='yellow', alpha=0.3)
ax.legend()
plt.show()

## Creating from an array

The `PupilMeasurement` can be built directly from a time and size array.

Replace the generated arrays with your real `time` and `size` data when ready.

In [ ]:
rng = np.random.default_rng(0)
time_data = np.linspace(0, 10, 1000)
size_data = 5 + np.sin(2 * np.pi * 0.5 * time_data) + rng.normal(0, 0.1, size=time_data.shape)

sin_pm = piprkit.PupilMeasurement(time_data=time_data, size_data=size_data)

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
sin_pm.plot(ax=ax)
plt.show()

## Existing arrays

The data can be extracted from the `PupilMeasurement` object using `get_time()` and `get_size()`. This returns regular numpy arrays, allowing easy integration with other workflows.

These arrays can easily, after optional editing, be used to recreate the `PupilMeasurement`.

This is useful when you have a pipeline that outputs arrays and you want to re-wrap them into `PupilMeasurement` for convenience.

In [ ]:
pm_blue = piprkit.load_simulated_pupil(blue_or_red='blue')

# We get the time data and pupil size data from the previous example, to work with more realistic data. This array can contain your real data.
time_data = pm_blue.get_time()
size_data = pm_blue.get_size()

# Let's show that it's a regular numpy array
print("Type and shape of arrays")
print(f"Type of time_data: {type(time_data)}, shape of time_data: {time_data.shape}")
print(f"Type of size_data: {type(size_data)}, shape of size_data: {size_data.shape}")
print("")
print("First few elements of data:")
print("Index: ", "".join([f"{i}\t" for i in range(5)]))
print("Time:  ", "".join([f"{time_data[i]:.3f}\t" for i in range(5)]))
print("Size:  ", "".join([f"{size_data[i]:.3f}\t" for i in range(5)]))
pm = piprkit.PupilMeasurement(time_data=time_data, size_data=size_data)

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
pm.plot(ax=ax)
plt.show()

## Setting and plotting light stimuli

The light stimulus can be added to the `PupilMeasurement` using `set_light_stimulus`. This light stimulus is used for a variety of functions, plots, and metrics.

In [ ]:
pm_blue = piprkit.load_simulated_pupil(blue_or_red='blue')
pm = piprkit.PupilMeasurement(time_data=pm_blue.get_time(), size_data=pm_blue.get_size())

# Setting light stimuli
pm.set_light_stimulus(start=-1, end=0)
print(pm.get_light_stimulus().get_timing())

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
pm.plot(ax=ax)
pm.plot_light_stimulus(ax=ax)
plt.show()

## Saving and loading CSV files

Using `piprkit.save_csv` the `PupilMeasurement` object can be saved. It will store a csv file with the time and size data, as well as a JSON sidecar with data on the blink, light stimuli, and more.

In [ ]:
measurement = piprkit.load_simulated_pupil(blue_or_red='blue')
measurement.set_blinks([(-4.0, -3.8)])

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "measurement.csv"
    piprkit.save_csv(measurement, path)
    print(path.with_suffix(".json").read_text())
    loaded = piprkit.load_csv(path)

print(type(loaded).__name__)
print(f"Light stimulus: {loaded.get_light_stimulus().get_timing()}")
print(f"Blinks: {loaded.get_blinks()}")

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
loaded.plot(ax=ax, color="blue", label="Loaded from CSV")
loaded.plot_light_stimulus(ax=ax)
loaded.plot_blinks(ax=ax, label="Blink")
ax.legend()
plt.show()